In [8]:
# Environment Setup & Automatic Directory Initialization
from google.colab import drive
import os

drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/adaptive_crypto_framework'

# Auto-create the directory structure required by the framework
subfolders = ['src', 'data/raw/sensitive', 'data/raw/normal', 'data/processed', 'notebooks', 'tests']
for folder in subfolders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

os.chdir(PROJECT_PATH)
print(f"✔ Workspace initialized. Current Directory: {os.getcwd()}")
print("👉 Action: Place your 3 sensitive CSVs in data/raw/sensitive/ and 3 normal CSVs in data/raw/normal/ before proceeding.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✔ Workspace initialized. Current Directory: /content/drive/MyDrive/adaptive_crypto_framework
👉 Action: Place your 3 sensitive CSVs in data/raw/sensitive/ and 3 normal CSVs in data/raw/normal/ before proceeding.


In [9]:
# Writing the Feature Extraction Module to src/monitor.py
%%writefile src/monitor.py
import math
from collections import Counter
import os

def calculate_shannon_entropy(data_str):
    """Evaluates character probability distributions to measure randomness density."""
    if not data_str:
        return 0
    total_len = len(data_str)
    counts = Counter(data_str)
    entropy = 0
    for count in counts.values():
        p = count / total_len
        entropy -= p * math.log2(p)
    return round(entropy, 3)

def extract_file_features(filename, content):
    """Transforms a raw textual string payload into a 4-dimensional numeric vector coordinate."""
    ext_map = {'.csv': 1, '.json': 2, '.db': 3, '.txt': 4}
    ext = '.' + filename.split('.')[-1] if '.' in filename else '.txt'
    ext_id = ext_map.get(ext, 0)

    size_kb = len(content.encode('utf-8')) / 1024
    entropy = calculate_shannon_entropy(content)

    compliance_keywords = ["medical", "patient", "ssn", "balance", "password", "bank", "cc_num", "glucose", "routing"]
    keyword_matches = sum(1 for word in compliance_keywords if word in content.lower())

    return [ext_id, size_kb, entropy, keyword_matches]

Overwriting src/monitor.py


In [10]:
# Executing the Unified Feature Processing Pipeline
import pandas as pd
import glob
import os
from src.monitor import extract_file_features

feature_matrix = []
labels = []

# Process Sensitive Bucket
sensitive_files = glob.glob('data/raw/sensitive/*.csv')
for filepath in sensitive_files:
    filename = os.path.basename(filepath)
    df = pd.read_csv(filepath, encoding='latin1', low_memory=False)
    for _, row in df.head(200).iterrows():
        feature_matrix.append(extract_file_features(filename, str(row.to_dict())))
        labels.append(1)

# Process Normal Bucket
normal_files = glob.glob('data/raw/normal/*.csv')
for filepath in normal_files:
    filename = os.path.basename(filepath)
    df = pd.read_csv(filepath, encoding='latin1', low_memory=False)
    for _, row in df.head(200).iterrows():
        feature_matrix.append(extract_file_features(filename, str(row.to_dict())))
        labels.append(0)

# Save Master Dataset
dataset = pd.DataFrame(feature_matrix, columns=['Ext_ID', 'Size_KB', 'Entropy', 'Keywords'])
dataset['Is_Sensitive'] = labels
dataset.to_csv('data/processed/sensitivity_dataset.csv', index=False)

print(f"✔ Pipeline successfully compiled {len(dataset)} balanced samples into data/processed/sensitivity_dataset.csv!")
print(dataset['Is_Sensitive'].value_counts().to_string())

✔ Pipeline successfully compiled 1200 balanced samples into data/processed/sensitivity_dataset.csv!
Is_Sensitive
1    600
0    600
